In [1]:
class Doc:
    def __init__(self, doc_id, skip_id=None):
        self.doc_id = doc_id # doc id
        self.skip_id = skip_id # skip pointer

class Record:
    def __init__(self, term_id, docs: dict):
        self.term_id = term_id # term id
        self.docs = docs # {doc_id: Doc}
        self.record = sorted(list(docs.keys())) # sorted doc ids
    
    @property
    def df(self):
        return len(self.record) # document frequency
    
    def i_th_doc(self, i):
        return self.docs[self.record[i]] # i-th document

In [2]:
class InvertedIndex:
    def __init__(self):
        self.records = dict()  # term_id -> Record

    def add(self, term_id, docs):
        if term_id not in self.records:  # new term
            self.records[term_id] = Record(term_id, docs)  # create new record

    def merge2(self, term_1, term_2, skip=False, _and=True):
        if term_1 in self.records and term_2 in self.records:  # both terms exist
            idx_1 = 0  # index of term_1
            idx_2 = 0  # index of term_2
            res = []  # result
            if _and:  # AND
                while (
                    idx_1 < self.records[term_1].df and 
                    idx_2 < self.records[term_2].df
                ):
                    doc_1 = self.records[term_1].i_th_doc(idx_1)  # doc of term_1
                    doc_2 = self.records[term_2].i_th_doc(idx_2)  # doc of term_2
                    if doc_1.doc_id == doc_2.doc_id:  # same doc
                        res.append(doc_1.doc_id)  # add to result
                        idx_1 += 1  # move to next doc of term_1
                        idx_2 += 1  # move to next doc of term_2
                    elif doc_1.doc_id < doc_2.doc_id:  # doc_1 < doc_2
                        if skip:  # use skip pointer
                            if (
                                doc_1.skip_id is not None
                                and doc_1.skip_id <= doc_2.doc_id
                            ):  # skip
                                while (
                                    self.records[term_1].record[idx_1] 
                                    < doc_1.skip_id
                                ):
                                    idx_1 += 1
                        else:  # no skip pointer
                            idx_1 += 1  # move to next doc of term_1
                    else:  # doc_1 > doc_2
                        if skip:  # use skip pointer
                            if (
                                doc_2.skip_id is not None
                                and doc_2.skip_id <= doc_1.doc_id
                            ):  # skip
                                while (
                                    self.records[term_2].record[idx_2] 
                                    < doc_2.skip_id
                                ):
                                    idx_2 += 1
                        else:  # no skip pointer
                            idx_2 += 1  # move to next doc of term_2
            else:  # OR
                while (
                    idx_1 < self.records[term_1].df and 
                    idx_2 < self.records[term_2].df
                ):
                    doc_1 = self.records[term_1].i_th_doc(idx_1)  # doc of term_1
                    doc_2 = self.records[term_2].i_th_doc(idx_2)  # doc of term_2
                    if doc_1.doc_id == doc_2.doc_id:  # same doc
                        res.append(doc_1.doc_id)  # add to result
                        idx_1 += 1  # move to next doc of term_1
                        idx_2 += 1  # move to next doc of term_2
                    elif doc_1.doc_id < doc_2.doc_id:  # doc_1 < doc_2
                        res.append(doc_1.doc_id)  # add to result
                        idx_1 += 1  # move to next doc of term_1
                    else:  # doc_1 > doc_2
                        res.append(doc_2.doc_id)  # add to result
                        idx_2 += 1  # move to next doc of term_2
                while idx_1 < self.records[term_1].df:  # term_1 has more docs
                    res.append(self.records[term_1].record[idx_1])  # add to result
                    idx_1 += 1  # move to next doc of term_1
                while idx_2 < self.records[term_2].df:  # term_2 has more docs
                    res.append(self.records[term_2].record[idx_2])  # add to result
                    idx_2 += 1  # move to next doc of term_2
            return res